# Figures 1 and 2
Run from the repository root after `reproduce.Rmd` creates `outputs/paper/figure_panel.csv`. Figures are saved to `outputs/paper/figures/`.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv("outputs/paper/figure_panel.csv")
df["RSSD9999"] = pd.PeriodIndex(df["RSSD9999"].str.replace(" ", "", regex=False), freq="Q").to_timestamp()
df = df.set_index(["RSSD9999", "RSSD9001"])


In [ ]:
import numpy as np
import scipy.stats as stats

In [ ]:
def yhatR2(x, y):
    reg = stats.linregress(x, y)
    xlin = np.linspace(min(x), max(x))
    yhat = xlin * reg[0] + reg[1]
    equation = f"y={reg[1]:.4f}{reg[0]:+.4f}x"
    return xlin, yhat, round(reg[2] ** 2, 2), equation

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(16, 13))
groupeddf = df.groupby(["RSSD9999"])[
    [
        "shortRateVol",
        "niiRateAnnSeasAdj",
        "nniRateAnnSeasAdj",
        "macroFinUnc12",
        "DGS10",
        "TP10",
        "AER10",
    ]
].mean()
groupeddf.shortRateVol = (
    groupeddf.shortRateVol - groupeddf.shortRateVol.mean()
) / groupeddf.shortRateVol.std()

ax[0, 0].scatter(groupeddf.shortRateVol, groupeddf.niiRateAnnSeasAdj)
ax[0, 0].set_ylabel("Net Interest Margin")
xlin, yhat, R2, equation = yhatR2(groupeddf.shortRateVol, groupeddf.niiRateAnnSeasAdj)
ax[0, 0].plot(xlin, yhat, linestyle="--", color="black", linewidth=2)
ax[0, 0].text(2.75, 0.0112, f"{equation}\nR-sqr: {R2}")

ax[0, 1].scatter(groupeddf.macroFinUnc12, groupeddf.niiRateAnnSeasAdj)
xlin, yhat, R2, equation = yhatR2(groupeddf.macroFinUnc12, groupeddf.niiRateAnnSeasAdj)
ax[0, 1].plot(xlin, yhat, linestyle="--", color="black", linewidth=2)
ax[0, 1].text(1.8, 0.0112, f"{equation}\nR-sqr: {R2}")


ax[1, 0].scatter(groupeddf.shortRateVol, groupeddf.nniRateAnnSeasAdj)
ax[1, 0].set_ylabel("Net Non-Interest Income")
ax[1, 0].set_xlabel("Short-Rate Standard Deviation")
xlin, yhat, R2, equation = yhatR2(groupeddf.shortRateVol, groupeddf.nniRateAnnSeasAdj)
ax[1, 0].plot(xlin, yhat, linestyle="--", color="black", linewidth=2)
ax[1, 0].text(2.75, -0.00500, f"{equation}\nR-sqr: {R2}")

ax[1, 1].scatter(groupeddf.macroFinUnc12, groupeddf.nniRateAnnSeasAdj)
ax[1, 1].set_xlabel("Macroeconomic Uncertainty")
xlin, yhat, R2, equation = yhatR2(groupeddf.macroFinUnc12, groupeddf.nniRateAnnSeasAdj)
ax[1, 1].plot(xlin, yhat, linestyle="--", color="black", linewidth=2)
ax[1, 1].text(1.8, -0.00500, f"{equation}\nR-sqr: {R2}")

fig.savefig("outputs/paper/figures/biVariatePlots.png", bbox_inches="tight")


In [ ]:
uncertainty_plot = groupeddf[["shortRateVol", "macroFinUnc12"]].rename(
    {
        "shortRateVol": "Short-Rate Standard Deviation",
        "macroFinUnc12": "Macroeconomic Uncertainty",
    },
    axis=1,
).plot(figsize=(17, 8), xlabel="Time")
uncertainty_plot.figure.savefig("outputs/paper/figures/uncertainPlots.png", bbox_inches="tight")
